In [1]:
import pandas as pd
import numpy as np
import pyarrow.dataset as ds
import s3fs

## Dataset Import

In [2]:
# Import FIP Dataset

s3_path_fip = (
    "s3://m3-intel-hub-dp-us-east-1-517292-prod/"
    "publish/data-product/financial_inventory_projection_report_network_update/"
)

dataset = ds.dataset(
    s3_path_fip,
    format="parquet",
    partitioning="hive" 
)

table = dataset.to_table(
    filter=(
        ds.field("date").isin(["202612"])  # "202712", "202812" 
    ) & 
    (
        ds.field("corporate_brand").isin(['ABRAXANE','REVLIMID',"All Other Pharmaceut"])
    ) &
    (
        ds.field("snapshot_date") >= "2025-10-01"
    ) & ~(
        (ds.field("snapshot_date") == "2026-01-23") &
        (ds.field("snapshot_type") == "friday")
    )
)


df_fip = table.to_pandas()
df_fip.head()

,material,plant,date,quantity,total_cost,concost_source,unit_of_measure,cost_per_unit,source,corporate_brand,material_type,development_lifecycle_status,enterprise_category,enterprise_sub_category,dosage_form_parent,corp_brand_id,network_or_business_unit,snapshot_type,snapshot_date
0,1457248,2061,202612,561.000,NaN,missing,None,NaN,rr,All Other Pharmaceut,UNBW,nan,nan,nan,nan,00201790,PHARMA,friday,2025-10-03
1,1457516,2061,202612,34000.000,NaN,missing,None,NaN,rr,All Other Pharmaceut,UNBW,nan,nan,nan,nan,00201790,PHARMA,friday,2025-10-03
2,1466493,2061,202612,4538.000,7.016565e+03,concost dp,ST,1.54618,rr,REVLIMID,PACK,COMMERCIAL,PACKAGE COMPONENT,PACKAGE COMPONENT,nan,03302101,PHARMA,friday,2025-10-03
3,1342328,1007,202612,54.043,3.147036e+06,concost dp,CYC,58232.08362,rr,All Other Pharmaceut,HALB,CLINICAL,OTHER PROCESS MATERIALS,OTHER PROCESS MATERIALS,nan,00201790,PHARMA,friday,2025-10-03
4,1431634,2028,202612,130.000,1.405270e+04,concost dp,ST,108.09767,rr,REVLIMID,FIN,COMMERCIAL,MARKET UNIT,SINGLE PRODUCT,CAPSULE,03302101,PHARMA,friday,2025-10-03


In [3]:
df_fip.shape

(79550, 19)

In [4]:
# Import Plant Type Data

s3_path_plants = (
    "s3://m3-intel-hub-dp-us-east-1-517292-prod/"
    "refined/data-asset/fin_inv_proj/"
    "bms_internal_vs_external_plants/"
    "bms_internal_vs_external_plants.parquet"
)

df_plants = pd.read_parquet(s3_path_plants)
#df_plants.head()

In [5]:
# Import node type Dataset

fs = s3fs.S3FileSystem()  # uses SageMaker execution role

parquet_files_nt = fs.glob(
    "s3://m3-intel-hub-dp-us-east-1-517292-prod/"
    "dbt_intelligence_hub/intelligence_hub_db_sandbox_staging/"
    "src__sap_t001w/data/*.parquet"
)

df_ntype = pd.read_parquet(
    parquet_files_nt,
    engine="pyarrow",
    dtype_backend="pyarrow",
    filesystem=fs
)

#df_ntype.head()

In [6]:
# Import Material Master Data

fs = s3fs.S3FileSystem()  # uses SageMaker execution role

parquet_files = fs.glob(
    "s3://m3-intel-hub-dp-us-east-1-517292-prod/"
    "refined/data-asset/fin_inv_proj/"
    "sap_material_master/data/*.parquet"
)

df_mm = pd.read_parquet(
    parquet_files,
    engine="pyarrow",
    dtype_backend="pyarrow",
    filesystem=fs
)

#df_mm.head()

In [7]:
# import boto3

# s3 = boto3.client("s3")

# bucket = "m3-intel-hub-dp-us-east-1-517292-prod"
# prefix = "dbt_intelligence_hub/intelligence_hub_db_sandbox_staging/src__sap_t001w"

# response = s3.list_objects_v2(
#     Bucket=bucket,
#     Prefix=prefix
# )

# if "Contents" in response:
#     for obj in response["Contents"]:
#         print(obj["Key"], obj["Size"])
# else:
#     print("No objects found or no access.")

## Data Prep

In [8]:
# Create has_non_zero flag at material–plant level

df_fip["has_non_zero"] = (
    df_fip
    .groupby(["material", "plant"])["total_cost"]
    .transform(lambda x: (x != 0).any())
    .astype(int)
)

# Apply the filter
df_fip = df_fip.loc[df_fip["has_non_zero"] == 1].drop(columns="has_non_zero")

In [9]:
df_fip.shape

(61790, 19)

In [10]:
base1 = df_fip.copy()

In [11]:
# Join material master
mm_cols = [
    "material_number",
    "plant",
    "profit_center",
    #"corporate_brand",
    "brand_name",
    #"corp_brand_id",
    "material_description",
    #"material_type",
    "material_group",
    "old_material_number",
    "base_unit_of_measure",
    "unit_of_weight",
    #"development_lifecycle_status",
    "plant_specific_material_status",
    "mrp_type",
    "procurement_type",
    "safety_stock",
    "minimum_lot_size",
    "maximum_lot_size",
    "fixed_lot_size",
    "total_replenishment_lead_time",
    "total_shelf_life",
    "batch_management",
    "abc_indicator",
    #"valuation_class",
    #"price_unit",
    #"price_control_indicator",
]

material_master_sel = (
    df_mm[mm_cols]
    .drop_duplicates(subset=["material_number", "plant"])
)


base1 = base1.merge(
    material_master_sel,
    left_on=["material", "plant"],
    right_on=["material_number", "plant"],
    how="left",
    validate="m:1" 
)
base1 = base1.drop(columns=["material_number"])

base1 = base1.merge(
    df_plants[["Plant", "Plant Type"]],
    left_on=["plant"],
    right_on=["Plant"],
    how="left"
).drop(columns=["Plant"])

node_type_lkp = (
    df_ntype[["werks", "nodetype"]]
    .drop_duplicates(subset=["werks"])
)

base1 = base1.merge(
    node_type_lkp,
    left_on="plant",
    right_on="werks",
    how="left",
    validate="m:1"
).drop(columns=["werks"])

base1.shape

(61790, 39)

In [12]:
# rearrange columns for better readability
new_cols = [
    'corporate_brand',
    'material_type', 'development_lifecycle_status', 'enterprise_category',
    'enterprise_sub_category', 'dosage_form_parent', 'corp_brand_id',
    'network_or_business_unit',
    'profit_center', 'brand_name', 'material_description', 'material_group',
    'old_material_number', 'base_unit_of_measure', 'unit_of_weight',
    'plant_specific_material_status', 'mrp_type', 'procurement_type',
    'safety_stock', 'minimum_lot_size', 'maximum_lot_size',
    'fixed_lot_size', 'total_replenishment_lead_time', 'total_shelf_life',
    'batch_management', 'abc_indicator',
    'source', 'concost_source', 'Plant Type', 'nodetype',
    'unit_of_measure',
    'material', 'plant', 'date', 'quantity', 'total_cost',
    'cost_per_unit', 'snapshot_type', 'snapshot_date'
]

base1 = base1[new_cols]


In [13]:
base1.columns

Index(['corporate_brand', 'material_type', 'development_lifecycle_status',
       'enterprise_category', 'enterprise_sub_category', 'dosage_form_parent',
       'corp_brand_id', 'network_or_business_unit', 'profit_center',
       'brand_name', 'material_description', 'material_group',
       'old_material_number', 'base_unit_of_measure', 'unit_of_weight',
       'plant_specific_material_status', 'mrp_type', 'procurement_type',
       'safety_stock', 'minimum_lot_size', 'maximum_lot_size',
       'fixed_lot_size', 'total_replenishment_lead_time', 'total_shelf_life',
       'batch_management', 'abc_indicator', 'source', 'concost_source',
       'Plant Type', 'nodetype', 'unit_of_measure', 'material', 'plant',
       'date', 'quantity', 'total_cost', 'cost_per_unit', 'snapshot_type',
       'snapshot_date'],
      dtype='object')

In [14]:
# Add material entry flag 

base = base1.copy()
first_seen = (
    base.groupby(["material", "plant","date"])["snapshot_date"]
      .transform("min")
)

base["sku_status"] = np.where(
    base["snapshot_date"] == first_seen,
    "NEW",
    "EXISTING"
)


In [15]:
# fip copy df for data prep
df = base.copy()
df["snapshot_date"] = pd.to_datetime(df["snapshot_date"])


# snapshot lookup table
snapshot_calendar = (
    df[["snapshot_type", "snapshot_date"]]
    .drop_duplicates()
    .sort_values(["snapshot_type", "snapshot_date"])
)


# attach prev snapshot to snapshot calendar
snapshot_calendar["prev_snapshot_date"] = (
    snapshot_calendar
    .groupby("snapshot_type")["snapshot_date"]
    .shift(1)
)
snapshot_calendar      # comparing bd13 - bd13 snapshots and friday-friday snapshots. no bd13-fri snapshots

,snapshot_type,snapshot_date,prev_snapshot_date
5844,bd13,2025-10-17,NaT
20863,bd13,2025-11-19,2025-10-17
36046,bd13,2025-12-17,2025-11-19
53919,bd13,2026-01-23,2025-12-17
0,friday,2025-10-03,NaT
2863,friday,2025-10-10,2025-10-03
8829,friday,2025-10-24,2025-10-10
11849,friday,2025-10-31,2025-10-24
14879,friday,2025-11-07,2025-10-31
17876,friday,2025-11-14,2025-11-07


In [16]:
# Attach previous snapshot date to each row

df = df.merge(
    snapshot_calendar[["snapshot_type", "snapshot_date", "prev_snapshot_date"]],
    on=["snapshot_type", "snapshot_date"],
    how="left"
)

In [17]:
# Prepare current and previous frames

# Current snapshot frame
current_df = df.copy()

current_df = current_df.rename(columns={
    "quantity": "quantity_curr",
    "cost_per_unit": "cost_per_unit_curr",
    "total_cost": "total_cost_curr",
})

# Previous snapshot frame
previous_df = df.rename(columns={
    "snapshot_date": "snapshot_date_prev",
    "quantity": "quantity_prev",
    "cost_per_unit": "cost_per_unit_prev",
    "total_cost": "total_cost_prev",
})[
    [
        "material",
        "plant",
        "date",
        "snapshot_type",
        "snapshot_date_prev",
        "quantity_prev",
        "cost_per_unit_prev",
        "total_cost_prev",
    ]
]


# Join current to previous snapshot
rca_base = current_df.merge(
    previous_df,
    left_on=[
        "material",
        "plant",
        "date",
        "snapshot_type",
        "prev_snapshot_date",
    ],
    right_on=[
        "material",
        "plant",
        "date",
        "snapshot_type",
        "snapshot_date_prev",
    ],
    how="left"
)

In [18]:
rca_base.shape

(61790, 45)

In [19]:
# Flags for material–plant presence 

# 1. Identify NEW in current snapshot
# Present now, but not present in previous snapshot
rca_base["is_new_in_current_snapshot"] = (
    rca_base["prev_snapshot_date"].notna() &
    rca_base["quantity_prev"].isna()
)


# 2. Identify DROPPED in current snapshot
# Present in previous snapshot but missing in current snapshot

# Identify valid previous snapshots (calendar-safe)
valid_prev_snapshots = (
    snapshot_calendar["prev_snapshot_date"]
        .dropna()
        .unique()
)

# Restrict previous snapshot data to valid transitions
previous_df_valid = previous_df[
    previous_df["snapshot_date_prev"].isin(valid_prev_snapshots)
]

# Anti-join: rows present in previous but missing in current
prev_only = previous_df_valid.merge(
    current_df[
        [
            "material",
            "plant",
            "date",
            "snapshot_type",
            "prev_snapshot_date",
        ]
    ],
    left_on=[
        "material",
        "plant",
        "date",
        "snapshot_type",
        "snapshot_date_prev",
    ],
    right_on=[
        "material",
        "plant",
        "date",
        "snapshot_type",
        "prev_snapshot_date",
    ],
    how="left",
    indicator=True
).query("_merge == 'left_only'")

# Mark dropped rows
prev_only["is_dropped_in_current_snapshot"] = True

# Ensure column alignment for concat
for col in rca_base.columns:
    if col not in prev_only.columns:
        prev_only[col] = np.nan

# Current snapshot rows are NOT dropped
rca_base["is_dropped_in_current_snapshot"] = False


# 3. Combine current + dropped rows

final_rca_frame = pd.concat(
    [rca_base, prev_only[rca_base.columns]],
    ignore_index=True
)

# 4. Normalize boolean flags

for col in [
    "is_new_in_current_snapshot",
    "is_dropped_in_current_snapshot",
]:
    final_rca_frame[col] = (
        final_rca_frame[col]
            .replace({1: True, 0: False})
            .fillna(False)
            .astype("boolean")
    )

/tmp/ipykernel_393262/3310939321.py:69: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_rca_frame = pd.concat(
/tmp/ipykernel_393262/3310939321.py:83: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


In [20]:
final_rca_frame.shape

(63576, 47)

In [21]:
snapshot_mapping_check = (
    final_rca_frame
    .loc[:, ["snapshot_type", "snapshot_date", "prev_snapshot_date"]]
    .drop_duplicates()
    .sort_values(["snapshot_type", "snapshot_date"])
)

snapshot_mapping_check

,snapshot_type,snapshot_date,prev_snapshot_date
5844,bd13,2025-10-17,NaT
20863,bd13,2025-11-19,2025-10-17
36046,bd13,2025-12-17,2025-11-19
53919,bd13,2026-01-23,2025-12-17
61969,bd13,NaT,NaT
0,friday,2025-10-03,NaT
2863,friday,2025-10-10,2025-10-03
8829,friday,2025-10-24,2025-10-10
11849,friday,2025-10-31,2025-10-24
14879,friday,2025-11-07,2025-10-31


Error handling

1. Division by zero & invalid math -
    Previous quantity = 0
    Previous cost = 0
    Previous FIP = 0
    Volatility = 0
2. Min rolling window
   less than min periods of 3
   Newly introduced SKU
   Flat history causing volatility = 0
3. double counting due to duplicates
4. missing prev snapshot data - nulls
   SKU appears for first time
   SKU disappears and reappears
   Previous quantity / cost not available
5. Extreme values in the history
    Very large quantities or costs in the past pushing the present numbers. The outliers in the past affects the current numbers. Exlcude the historical outliers
   

# Data Transformations

### Driver Calculations

In [22]:
# Step 1: Compute raw change metrics (always runs)

# Base deltas

final_rca_frame["delta_quantity"] = (
    final_rca_frame["quantity_curr"] - final_rca_frame["quantity_prev"]
)

final_rca_frame["delta_cost_per_unit"] = (
    final_rca_frame["cost_per_unit_curr"] - final_rca_frame["cost_per_unit_prev"]
)


# Impact decomposition

# Quantity Impact 
final_rca_frame["quantity_impact"] = (
    final_rca_frame["delta_quantity"] *
    final_rca_frame["cost_per_unit_prev"]
)

# Cost Impact
final_rca_frame["cost_impact"] = (
    final_rca_frame["delta_cost_per_unit"] * final_rca_frame["quantity_prev"]
)

# Intercation
final_rca_frame["interaction_impact"] = (
    final_rca_frame["delta_quantity"] *
    final_rca_frame["delta_cost_per_unit"]
)

# Total Change
final_rca_frame["total_fip_change"] = (
    final_rca_frame["quantity_impact"] +
    final_rca_frame["cost_impact"] +
    final_rca_frame["interaction_impact"]
)

# Core Metrics

# delta_quantity_pct 
final_rca_frame["delta_quantity_pct"] = np.where(
    final_rca_frame["quantity_prev"] > 0,
    final_rca_frame["delta_quantity"] / final_rca_frame["quantity_prev"],
    np.nan
)

# delta_cost_per_unit_pct

final_rca_frame["delta_cost_per_unit_pct"] = np.where(
    final_rca_frame["cost_per_unit_prev"] > 0,
    final_rca_frame["delta_cost_per_unit"] / final_rca_frame["cost_per_unit_prev"],
    np.nan
)


# Contribution shares (absolute, normalized)
impact_abs_sum_qc = (
    final_rca_frame["quantity_impact"].abs() +
    final_rca_frame["cost_impact"].abs()
)

impact_abs_sum_all = (
    impact_abs_sum_qc +
    final_rca_frame["interaction_impact"].abs()
)


# quantity_impact_pct_of_total 
final_rca_frame["quantity_impact_pct_of_total"] = np.where(
    impact_abs_sum_qc > 0,
    final_rca_frame["quantity_impact"].abs() / impact_abs_sum_qc,
    0
)

# cost_impact_pct_of_total 
final_rca_frame["cost_impact_pct_of_total"] = np.where(
    impact_abs_sum_qc > 0,
    final_rca_frame["cost_impact"].abs() / impact_abs_sum_qc,
    0
)

# interaction_pct
final_rca_frame["interaction_pct"] = np.where(
    impact_abs_sum_all > 0,
    final_rca_frame["interaction_impact"].abs() / impact_abs_sum_all,
    0
)


# Dominance Score

final_rca_frame["abs_qty_impact"] = final_rca_frame["quantity_impact"].abs()
final_rca_frame["abs_cost_impact"] = final_rca_frame["cost_impact"].abs()

final_rca_frame["dominance_score"] = np.where(
    (final_rca_frame["abs_qty_impact"] + final_rca_frame["abs_cost_impact"]) == 0,
    0.0,  # avoid divide-by-zero → treat as neutral
    (final_rca_frame["abs_qty_impact"] - final_rca_frame["abs_cost_impact"]) /
    (final_rca_frame["abs_qty_impact"] + final_rca_frame["abs_cost_impact"])
)

### Data Sufficiency Metrics

In [23]:
final_rca_frame = final_rca_frame.sort_values(
    ["material", "plant","date", "snapshot_date"]
)

final_rca_frame["is_qty_present_greater0"] = (
    final_rca_frame["quantity_curr"] > 0
).astype(int)

# 1) history_weeks_count
final_rca_frame["history_weeks_count_greater0"] = (
    final_rca_frame
    .groupby(["material", "plant","date"])["is_qty_present_greater0"]
    .rolling(window=12, min_periods=1)
    .sum()
    .reset_index(level=[0,1,2], drop=True)
)

final_rca_frame["hist_count"] = (
    final_rca_frame
        .groupby(["material", "plant", "date"])["quantity_curr"]
        .transform(lambda x: x.notna().cumsum() - 1)
)

final_rca_frame["has_sufficient_6periods"] = (
    final_rca_frame["hist_count"] >= 6
)


# 2) presence_stability_score
final_rca_frame["presence_stability_score"] = (
    final_rca_frame["hist_count"] / 12
).clip(upper=1.0)

# 3) History confidence (intentionally mirrors stability)
final_rca_frame["history_confidence"] = (
    final_rca_frame["hist_count"] / 12
).clip(upper=1.0)


# 4) sku_presence_class
conditions = [
    final_rca_frame["is_new_in_current_snapshot"] == True,  
    final_rca_frame["is_dropped_in_current_snapshot"] == True,
    (
        (final_rca_frame["quantity_prev"] == 0) &
        (final_rca_frame["quantity_curr"] > 0) &
        (final_rca_frame["history_weeks_count_greater0"] < 6)
    ),
    (
        (final_rca_frame["presence_stability_score"] >= 0.75) &
        (final_rca_frame["history_weeks_count_greater0"] >= 6)
    )
]
 
choices = [
    "ENTERED",
    "OUT",
    "REACTIVATED",
    "STABLE"
]
 
final_rca_frame["sku_presence_class"] = np.select(
    conditions,
    choices,
    default="ERRATIC"
)


# 5) uom_change_flag
final_rca_frame = final_rca_frame.sort_values(
    ["material", "plant","date", "snapshot_date"]
)

final_rca_frame["unit_of_measure_prev"] = (
    final_rca_frame
    .groupby(["material", "plant","date"])["base_unit_of_measure"]
    .shift(1)
)

final_rca_frame["uom_change_flag"] = (
    final_rca_frame["unit_of_measure_prev"].notna() &
    (final_rca_frame["unit_of_measure_prev"] != final_rca_frame["unit_of_measure"])
)


# 6) duplicate_sku_plant_flag
dup_counts = (
    final_rca_frame
    .groupby(["material", "plant","date", "snapshot_date"])
    .size()
    .rename("dup_count")
    .reset_index()
)
final_rca_frame = final_rca_frame.merge(
    dup_counts,
    on=["material", "plant","date", "snapshot_date"],
    how="left"
)
final_rca_frame["duplicate_sku_plant_flag"] = (
    final_rca_frame["dup_count"] > 1
)


# 7) dominance_instability_flag

final_rca_frame["dominance_score_var_4"] = (
    final_rca_frame
    .groupby(["material", "plant","date"])["dominance_score"]
    .rolling(window=4, min_periods=2)
    .var()
    .reset_index(level=[0,1,2], drop=True)
)
final_rca_frame["dominance_instability_flag"] = (
    final_rca_frame["dominance_score_var_4"] > 0.25
)


# 8) RCA Mode

conditions = [
    # 1) No RCA: UOM change or duplicate SKU–plant
    (
        final_rca_frame["uom_change_flag"].fillna(False).astype(bool) |
        final_rca_frame["duplicate_sku_plant_flag"].fillna(False).astype(bool)
    ),
    # 2) Entry / Exit RCA
    (
        final_rca_frame["sku_presence_class"]
        .isin(["ENTERED", "OUT"])
        .fillna(False)
    ),
    # 3) Limited RCA: insufficient history
    (
        ~final_rca_frame["has_sufficient_6periods"]
    ),
    # 4) Full temporal RCA: stable SKU
    (
        final_rca_frame["sku_presence_class"]
        .eq("STABLE")
        .fillna(False)
    )
]

choices = [
    "NO_RCA",
    "ENTRY_EXIT_RCA",
    "LIMITED_RCA",
    "FULL_TEMPORAL_RCA"
]

final_rca_frame["rca_mode"] = np.select(
    conditions,
    choices,
    default="LIMITED_RCA"
)

### Noise vs Signal Determination 

In [24]:
# Sorting

final_rca_frame = final_rca_frame.sort_values(
    ["material", "plant","date", "snapshot_date"]
)

#  1) Business materiality (value-based)   
# How big the change is in value terms, relative to prior fip.
final_rca_frame["relative_fip_impact"] = np.where(
    final_rca_frame["total_cost_prev"] > 0,
    final_rca_frame["total_fip_change"].abs() /
    final_rca_frame["total_cost_prev"],
    np.nan
)

#### Persistance

In [25]:
# Temporal persistence (directional consistency)

final_rca_frame["quantity_change_sign"] = np.sign(
    final_rca_frame["delta_quantity"]
) 

def persistence_score(series):
    score = []
    current = 0
    prev = 0
    for v in series:
        if v == 0 or pd.isna(v):
            current = 0
        elif v == prev:
            current += 1
        else:
            current = 1
        score.append(current)
        prev = v
    return score

final_rca_frame["quantity_persistence_score"] = (
    final_rca_frame
    .groupby(["material", "plant","date"])["quantity_change_sign"]
    .transform(persistence_score)
)


In [26]:
# Outlier Identification

# PARAMETERS

ROLLING_WINDOW = 12
MIN_PERIODS = 6

EXTREME_PCT_CHANGE = 0.5      # 50%
LOW_LEVEL_FLOOR = 0.05        # collapse threshold (5%)
HIGH_LEVEL_MULT = 10         # spike threshold (10x)

ROBUST_Z_THRESHOLD = 3
MIN_ZSCORE_POINTS = 3
PCTL_FALLBACK = 0.95

# 1) Rolling median of quantity (baseline scale)

final_rca_frame["quantity_rolling_median_12w"] = (
    final_rca_frame
        .groupby(["material", "plant","date"])["quantity_curr"]
        .transform(
            lambda x: x.shift(1)
                      .rolling(ROLLING_WINDOW, min_periods=MIN_PERIODS)
                      .median()
        )
)

# 2) Structural extreme change (scale-based)

final_rca_frame["is_structural_extreme_qty"] = (
    (final_rca_frame["delta_quantity_pct"].abs() >= EXTREME_PCT_CHANGE) &
    (
        (final_rca_frame["quantity_curr"] <=
         LOW_LEVEL_FLOOR * final_rca_frame["quantity_rolling_median_12w"]) |
        (final_rca_frame["quantity_curr"] >=
         HIGH_LEVEL_MULT * final_rca_frame["quantity_rolling_median_12w"])
    )
)



# 3) CLEAN delta series (absolute, exclude structural extremes)

final_rca_frame["clean_delta_quantity"] = final_rca_frame["delta_quantity"]

final_rca_frame.loc[
    final_rca_frame["is_structural_extreme_qty"],
    "clean_delta_quantity"
] = np.nan



# 4) Robust rolling MAD of absolute delta quantity

final_rca_frame["delta_qty_rolling_mad_12w"] = (
    final_rca_frame
        .groupby(["material", "plant","date"])["clean_delta_quantity"]
        .transform(
            lambda x: x.shift(1)
                      .rolling(ROLLING_WINDOW, min_periods=MIN_PERIODS)
                      .apply(
                          lambda s: np.nanmedian(
                              np.abs(s - np.nanmedian(s))
                          ),
                          raw=True
                      )
        )
)

# 5) Robust MAD-based z-score (absolute delta)

def robust_zscore(series, min_points=MIN_ZSCORE_POINTS):
    valid = series.dropna()

    if len(valid) < min_points:
        return pd.Series(np.nan, index=series.index)

    median = np.nanmedian(valid)
    mad = np.nanmedian(np.abs(valid - median))

    if mad == 0 or np.isnan(mad):
        return pd.Series(np.nan, index=series.index)

    return (series - median) / (1.4826 * mad)


final_rca_frame["quantity_z_scr"] = (
    final_rca_frame
        .groupby(["material", "plant","date"])["clean_delta_quantity"]
        .transform(lambda x: robust_zscore(x.shift(1)))
)



# 6) Z-score availability (CORRECT, LOCAL gating)

final_rca_frame["qty_z_hist_count"] = (
    final_rca_frame
        .groupby(["material", "plant","date"])["clean_delta_quantity"]
        .transform(
            lambda x: x.shift(1)
                      .rolling(ROLLING_WINDOW, min_periods=1)
                      .count()
        )
)

final_rca_frame["has_quantity_zscore"] = (
    (final_rca_frame["qty_z_hist_count"] >= MIN_ZSCORE_POINTS) &
    (final_rca_frame["delta_qty_rolling_mad_12w"] > 0)
)



# 7) Statistical outlier (MAD-based)

final_rca_frame["is_statistical_outlier_qty"] = (
    final_rca_frame["has_quantity_zscore"] &
    (final_rca_frame["quantity_z_scr"].abs() > ROBUST_Z_THRESHOLD)
)


# 8) Percentile fallback (ONLY when z-score unavailable)
final_rca_frame["qty_abs_pct_threshold"] = (
    final_rca_frame
        .groupby(["material", "plant","date"])["clean_delta_quantity"]
        .transform(
            lambda x: x.shift(1)
                      .abs()
                      .quantile(PCTL_FALLBACK)
        )
)

# Fallback scale must be valid (non-zero)
final_rca_frame["has_valid_pct_scale"] = (
    final_rca_frame["qty_abs_pct_threshold"] > 0
)

final_rca_frame["is_pct_outlier_qty"] = (
    final_rca_frame["clean_delta_quantity"].abs() >
    final_rca_frame["qty_abs_pct_threshold"]
)

# 9) FINAL quantity outlier flag (hierarchical & SAFE)

final_rca_frame["is_quantity_outlier"] = (
    final_rca_frame["is_structural_extreme_qty"] |
    np.where(
        final_rca_frame["has_quantity_zscore"],
        final_rca_frame["is_statistical_outlier_qty"],
        final_rca_frame["has_valid_pct_scale"] &
        final_rca_frame["is_pct_outlier_qty"]
    )
)


#### Change Point Detection

In [27]:
# Change Point Detection

# PARAMETERS

CP_LONG_WINDOW = 12
CP_LONG_MIN = 6
CP_SHORT_WINDOW = 6
CP_SHORT_MIN = 3

CP_TREND_STRENGTH_THRESHOLD = 2
CP_FALLBACK_STRENGTH_THRESHOLD = 5   # conservative
MIN_PERSISTENCE_CP = 2


# 1) Rolling average of ABSOLUTE delta quantity (long window)

final_rca_frame["rolling_avg_delta_qty_12w"] = (
    final_rca_frame
        .groupby(["material", "plant","date"])["delta_quantity"]
        .transform(
            lambda x: x.shift(1)
                      .rolling(CP_LONG_WINDOW, min_periods=CP_LONG_MIN)
                      .mean()
        )
)


# 2) rolling MAD of ABSOLUTE delta quantity

final_rca_frame["delta_qty_rolling_mad_12w"] = (
    final_rca_frame
        .groupby(["material", "plant","date"])["delta_quantity"]
        .transform(
            lambda x: x.shift(1)
                      .rolling(CP_LONG_WINDOW, min_periods=CP_LONG_MIN)
                      .apply(
                          lambda s: np.nanmedian(
                              np.abs(s - np.nanmedian(s))
                          ),
                          raw=True
                      )
        )
)

# 3) Primary trend strength (MAD-based)

final_rca_frame["trend_strength"] = (
    final_rca_frame["rolling_avg_delta_qty_12w"].abs() /
    (1.4826 * final_rca_frame["delta_qty_rolling_mad_12w"])
)

# Invalidate degenerate cases
final_rca_frame.loc[
    (final_rca_frame["delta_qty_rolling_mad_12w"] <= 0) |
    (final_rca_frame["delta_qty_rolling_mad_12w"].isna()),
    "trend_strength"
] = np.nan

final_rca_frame["has_valid_cp_strength"] = (
    final_rca_frame["trend_strength"].notna()
)


# 4) GENERIC fallback scale (never collapses)

final_rca_frame["delta_qty_rolling_median_abs_12w"] = (
    final_rca_frame
        .groupby(["material", "plant","date"])["delta_quantity"]
        .transform(
            lambda x: x.shift(1)
                      .rolling(CP_LONG_WINDOW, min_periods=CP_LONG_MIN)
                      .apply(lambda s: np.nanmedian(np.abs(s)), raw=True)
        )
)

final_rca_frame["fallback_change_strength"] = (
    final_rca_frame["rolling_avg_delta_qty_12w"].abs() /
    final_rca_frame["delta_qty_rolling_median_abs_12w"]
)

# Invalidate fallback when scale unusable
final_rca_frame.loc[
    (final_rca_frame["delta_qty_rolling_median_abs_12w"] <= 0) |
    (final_rca_frame["delta_qty_rolling_median_abs_12w"].isna()),
    "fallback_change_strength"
] = np.nan

final_rca_frame["use_fallback_cp"] = (
    final_rca_frame["trend_strength"].isna() &
    final_rca_frame["fallback_change_strength"].notna()
)

# 5) Short-window trend confirmation (ABSOLUTE delta)
final_rca_frame["rolling_avg_delta_qty_6w"] = (
    final_rca_frame
        .groupby(["material", "plant","date"])["delta_quantity"]
        .transform(
            lambda x: x.shift(1)
                      .rolling(CP_SHORT_WINDOW, min_periods=CP_SHORT_MIN)
                      .mean()
        )
)

final_rca_frame["trend_confirmed"] = (
    np.sign(final_rca_frame["rolling_avg_delta_qty_12w"]) ==
    np.sign(final_rca_frame["rolling_avg_delta_qty_6w"])
)


# 6) FINAL generic change-point detection
final_rca_frame["change_point_detected"] = (
    (final_rca_frame["quantity_persistence_score"] >= MIN_PERSISTENCE_CP) &
    final_rca_frame["trend_confirmed"] &
    (
        # Primary MAD-based path
        (
            final_rca_frame["has_valid_cp_strength"] &
            (final_rca_frame["trend_strength"] > CP_TREND_STRENGTH_THRESHOLD)
        )
        |
        # Fallback absolute-scale path
        (
            final_rca_frame["use_fallback_cp"] &
            (final_rca_frame["fallback_change_strength"] > CP_FALLBACK_STRENGTH_THRESHOLD)
        )
    )
)



#### Regime Stability

In [28]:
# Regime stability - validate the change point with stability

REGIME_STABILITY_WINDOW = 3        # how many snapshots must settle
REGIME_STABILITY_TOLERANCE = 0.2   # ±20% band around new level

# 1) Reference "new level" after change

final_rca_frame["post_cp_level"] = (
    final_rca_frame
        .groupby(["material", "plant","date"])["quantity_curr"]
        .shift(1)
)


# 2) Within-band check
final_rca_frame["within_new_regime_band"] = (
    final_rca_frame["quantity_curr"].between(
        final_rca_frame["post_cp_level"] * (1 - REGIME_STABILITY_TOLERANCE),
        final_rca_frame["post_cp_level"] * (1 + REGIME_STABILITY_TOLERANCE)
    )
)


# 3) Rolling stability confirmation
final_rca_frame["regime_stability_score"] = (
    final_rca_frame
        .groupby(["material", "plant","date"])["within_new_regime_band"]
        .transform(
            lambda x: x.shift(-1)   # look forward (post-change validation)
                      .rolling(REGIME_STABILITY_WINDOW, min_periods=REGIME_STABILITY_WINDOW)
                      .sum()
        )
)

final_rca_frame["is_regime_stable"] = (
    final_rca_frame["regime_stability_score"] >= REGIME_STABILITY_WINDOW
)


final_rca_frame["effective_change_point"] = (
    final_rca_frame["change_point_detected"] &
    final_rca_frame["is_regime_stable"]
)


#### Noise & Signal Flagging

In [29]:
# BASE METHOD 

final_rca_frame["is_noise_base"] = (
    final_rca_frame["is_quantity_outlier"] &
    (~final_rca_frame["effective_change_point"])
)

# insufficient data → cannot classify as noise yet
final_rca_frame.loc[
    ~final_rca_frame["has_sufficient_6periods"],
    "is_noise_base"
] = False

final_rca_frame["is_signal_base"] = ~final_rca_frame["is_noise_base"]


In [30]:
# 2nd Layer Flaging

final_rca_frame["is_explainable"] = (
    # data quality must be OK
    (~final_rca_frame["uom_change_flag"]) &
    (~final_rca_frame["duplicate_sku_plant_flag"]) 
    # &
    # # must not be lifecycle noise -- needs data backed thresholds to handle different edge cases
    # (final_rca_frame["sku_presence_class"] == "STABLE")
)

final_rca_frame["is_noise_governed"] = (
    final_rca_frame["is_noise_base"] |
    (~final_rca_frame["is_explainable"])
)

final_rca_frame["is_signal_governed"] = (
    ~final_rca_frame["is_noise_governed"]
)

final_rca_frame["noise_reason"] = None

final_rca_frame.loc[
    final_rca_frame["uom_change_flag"] |
    final_rca_frame["duplicate_sku_plant_flag"],
    "noise_reason"
] = "DATA_QUALITY"

final_rca_frame.loc[
    final_rca_frame["noise_reason"].isna() &
    final_rca_frame["is_noise_base"],
    "noise_reason"
] = "STATISTICAL_NOISE"


## Price Outliers Addition

In [31]:
PRICE_OUTLIER_PCT_THRESHOLD = 1   # 100% change

final_rca_frame["is_price_outlier"] = (
    final_rca_frame["delta_cost_per_unit_pct"].abs() >= PRICE_OUTLIER_PCT_THRESHOLD
)

final_rca_frame["price_event_type"] = "NO_PRICE_EVENT"

# Price-driven RCA case (quantity is NOT the driver)
final_rca_frame.loc[
    final_rca_frame["is_signal_governed"] &
    final_rca_frame["is_price_outlier"] &
    (~final_rca_frame["is_quantity_outlier"]),
    "price_event_type"
] = "PRICE_PRIMARY"

# Mixed RCA case (price amplifies a quantity-driven change)
final_rca_frame.loc[
    final_rca_frame["is_signal_governed"] &
    final_rca_frame["is_price_outlier"] &
    final_rca_frame["is_quantity_outlier"],
    "price_event_type"
] = "PRICE_SECONDARY"


#### Brand + Snapshot Date Level Top 10 SKUs

In [32]:
final_rca_frame["abs_fip_change"] = final_rca_frame["total_fip_change"].abs()

final_rca_frame["snapshot_signal_rank"] = (
    final_rca_frame
    .where(final_rca_frame["is_signal_governed"])
    .groupby(["snapshot_date", "corporate_brand","date"])["abs_fip_change"]
    .rank(method="first", ascending=False)
)
final_rca_frame["is_top10_contributor"] = (
    final_rca_frame["snapshot_signal_rank"] <= 10
)

final_rca_frame = final_rca_frame.drop(["snapshot_signal_rank","abs_fip_change"],axis=1)

In [33]:
final_rca_frame.head()

,corporate_brand,material_type,development_lifecycle_status,enterprise_category,enterprise_sub_category,dosage_form_parent,corp_brand_id,network_or_business_unit,profit_center,brand_name,...,effective_change_point,is_noise_base,is_signal_base,is_explainable,is_noise_governed,is_signal_governed,noise_reason,is_price_outlier,price_event_type,is_top10_contributor
0,All Other Pharmaceut,PACK,COMMERCIAL,PACKAGE COMPONENT,PACKAGE COMPONENT,nan,00201790,PHARMA,11322,EXCIPIENT COMPONENTS,...,False,False,True,True,False,True,None,False,NO_PRICE_EVENT,False
1,All Other Pharmaceut,PACK,COMMERCIAL,PACKAGE COMPONENT,PACKAGE COMPONENT,nan,00201790,PHARMA,11322,EXCIPIENT COMPONENTS,...,False,False,True,<NA>,<NA>,<NA>,None,False,NO_PRICE_EVENT,False
2,All Other Pharmaceut,PACK,COMMERCIAL,PACKAGE COMPONENT,PACKAGE COMPONENT,nan,00201790,PHARMA,11322,EXCIPIENT COMPONENTS,...,False,False,True,<NA>,<NA>,<NA>,None,False,NO_PRICE_EVENT,False
3,All Other Pharmaceut,PACK,COMMERCIAL,PACKAGE COMPONENT,PACKAGE COMPONENT,nan,00201790,PHARMA,11322,EXCIPIENT COMPONENTS,...,False,False,True,<NA>,<NA>,<NA>,None,False,NO_PRICE_EVENT,False
4,All Other Pharmaceut,PACK,COMMERCIAL,PACKAGE COMPONENT,PACKAGE COMPONENT,nan,00201790,PHARMA,11322,EXCIPIENT COMPONENTS,...,False,False,True,<NA>,<NA>,<NA>,None,False,NO_PRICE_EVENT,False


In [34]:
# brands_to_keep = [
#     "All Other Pharmaceut"
#     # ,
#     #"ABRAXANE"
#     #,
#     #"REVLIMID"
# ]
# dates_to_keep = ["202612"] #, "202712", "202812"]

# filtered_df = final_rca_frame[
#     final_rca_frame["corporate_brand"].isin(brands_to_keep) &
#     final_rca_frame["date"].isin(dates_to_keep)
# ]

In [35]:
# final_rca_frame.loc[
#     final_rca_frame["corporate_brand"].str.contains(
#         "All Other Pharmaceut", case=False, na=False
#     ),
#     "corporate_brand"
# ].unique()


In [36]:
final_rca_frame.columns

Index(['corporate_brand', 'material_type', 'development_lifecycle_status',
       'enterprise_category', 'enterprise_sub_category', 'dosage_form_parent',
       'corp_brand_id', 'network_or_business_unit', 'profit_center',
       'brand_name',
       ...
       'effective_change_point', 'is_noise_base', 'is_signal_base',
       'is_explainable', 'is_noise_governed', 'is_signal_governed',
       'noise_reason', 'is_price_outlier', 'price_event_type',
       'is_top10_contributor'],
      dtype='object', length=113)

# Brand & Material Type Level Aggregations

### Data Prep

In [43]:
sku_df = final_rca_frame.copy()

# Define SKU correctly: material–plant
sku_df["sku_id"] = (
    sku_df["material"].astype(str) + "||" +
    sku_df["plant"].astype(str)
)

# Normalize boolean (critical for stability)
sku_df["is_signal_governed"] = (
    sku_df["is_signal_governed"]
        .fillna(False)
        .astype(bool)
)

# Helper columns
sku_df["abs_sku_impact"] = sku_df["total_fip_change"].abs()

sku_df["signal_impact"] = np.where(
    sku_df["is_signal_governed"],
    sku_df["abs_sku_impact"],
    0.0
)

sku_df["conflict_evaluable"] = (
    sku_df["is_signal_governed"] &
    sku_df["quantity_impact"].notna() &
    sku_df["cost_impact"].notna()
)

sku_df["driver_conflict"] = np.where(
    sku_df["conflict_evaluable"],
    np.sign(sku_df["quantity_impact"]) != np.sign(sku_df["cost_impact"]),
    np.nan
)

sku_df["weighted_dom_component"] = (
    sku_df["dominance_score"] * sku_df["abs_sku_impact"]
)

sku_df["signal_structural_cp"] = (
    sku_df["is_signal_governed"] &
    sku_df["effective_change_point"]
)


In [44]:
sku_df.shape

(63576, 120)

# Hierarchy based Sorting

In [45]:
# # 1. Hierarchy definition

# HIERARCHY = [
#     ["material_type"],
#     ["dosage_form_parent"],
#     ["material_group"],
#     ["material"],
#     ["nodetype"],
#     ["Plant Type"],
#     ["plant"],
#     ["material", "plant"]  # material–plant (most granular)
# ]

# # 2. Compute explainability at a level (SKU table ONLY)

# def compute_explainability(sku_brand_df, group_cols):
#     """
#     Explainability = share of brand's total absolute impact
#     explained by the top contributor at this level.
#     """

#     group_keys = ["date", "snapshot_date"] + group_cols

#     grp = (
#         sku_brand_df
#         .groupby(group_keys, as_index=False)
#         .agg(level_abs_impact=("abs_sku_impact", "sum"))
#     )

#     total_impact = sku_brand_df["abs_sku_impact"].sum()

#     if total_impact == 0 or grp.empty:
#         return 0.0, {}

#     top_row = grp.loc[grp["level_abs_impact"].idxmax()]
#     top_impact = top_row["level_abs_impact"]

#     explainability = top_impact / total_impact

#     top_entity = {col: top_row[col] for col in group_cols}

#     return explainability, top_entity


# # 3. Walk hierarchy & find dominant level

# def find_dominant_level(sku_brand_df):
#     """
#     Walks hierarchy and selects the level
#     that provides the maximum marginal clarity gain.
#     """

#     results = []
#     prev_explainability = 0.0

#     for level in HIERARCHY:
#         explainability, entity = compute_explainability(sku_brand_df, level)

#         marginal_gain = explainability - prev_explainability

#         results.append({
#             "level": " × ".join(level),
#             "explainability": explainability,
#             "marginal_gain": marginal_gain,
#             "entity": entity
#         })

#         prev_explainability = explainability

#     results_df = pd.DataFrame(results)

#     best_row = results_df.loc[results_df["marginal_gain"].idxmax()]

#     return {
#         "dominant_level": best_row["level"],
#         "dominant_entity": best_row["entity"],
#         "dominant_explainability": best_row["explainability"],
#         "full_trace": results_df
#     }


# # 4. Apply per brand + date + snapshot

# dominant_results = []

# group_keys = ["corporate_brand","corp_brand_id", "date", "snapshot_date"]

# for (brand, brand_id, date, snapshot_date), sku_brand_df in sku_df.groupby(group_keys):

#     result = find_dominant_level(sku_brand_df)

#     dominant_results.append({
#         "corporate_brand": brand,
#         "corp_brand_id": brand_id,
#         "date": date,
#         "snapshot_date": snapshot_date,
#         "dominant_level": result["dominant_level"],
#         "dominant_entity": result["dominant_entity"],
#         "dominant_explainability": result["dominant_explainability"]
#     })

# brand_dominance_df = pd.DataFrame(dominant_results)

# brand_dominance_df.head()


# Brand Level Calculations

In [46]:
# 1. BRAND-LEVEL BASE AGG

brand_df = (
    sku_df
    .groupby(
        ["corp_brand_id", "corporate_brand", "date", "snapshot_date"],
        as_index=False
    )
    .agg(
        ΔFIP=("total_fip_change", "sum"),
        Prior_FIP=("total_cost_prev", "sum"),
        Abs_Impact=("abs_sku_impact", "sum"),

        Total_SKUs=("sku_id", "nunique"),
        Signal_SKUs=("is_signal_governed", "sum"),
        Signal_Impact=("signal_impact", "sum"),

        Qty_Impact=("quantity_impact", "sum"),
        Cost_Impact=("cost_impact", "sum"),

        Weighted_Dom_Num=("weighted_dom_component", "sum"),
        Weighted_Dom_Den=("abs_sku_impact", "sum"),

        Driver_Conflict_Count=("driver_conflict", "sum"),
        Avg_Persistence=("quantity_persistence_score", "mean"),
        Structural_CP_Count=("signal_structural_cp", "sum"),
    )
)


# 2. SAFE DIVISION HELPERS


def safe_div(n, d):
    return np.where(d != 0, n / d, np.nan)


# 3. DERIVED BRAND METRICS


brand_df["FIP_pct_change"] = safe_div(
    brand_df["ΔFIP"],
    brand_df["Prior_FIP"]
)

brand_df["Net_vs_Abs_Ratio"] = safe_div(
    brand_df["ΔFIP"].abs(),
    brand_df["Abs_Impact"]
)

brand_df["Signal_SKU_%"] = safe_div(
    brand_df["Signal_SKUs"],
    brand_df["Total_SKUs"]
)

brand_df["Impact_from_Signal_%"] = safe_div(
    brand_df["Signal_Impact"],
    brand_df["Abs_Impact"]
)

brand_df["Qty_Impact_%"] = safe_div(
    brand_df["Qty_Impact"],
    brand_df["ΔFIP"]
)

brand_df["Cost_Impact_%"] = safe_div(
    brand_df["Cost_Impact"],
    brand_df["ΔFIP"]
)

brand_df["Weighted_Dominance"] = safe_div(
    brand_df["Weighted_Dom_Num"],
    brand_df["Weighted_Dom_Den"]
)

brand_df["Driver_Conflict_%"] = safe_div(
    brand_df["Driver_Conflict_Count"],
    brand_df["Signal_SKUs"]
)

brand_df["Structural_Shift_Index"] = safe_div(
    brand_df["Structural_CP_Count"],
    brand_df["Signal_SKUs"]
) * 100

brand_df["Ownership_Clarity_Index"] = (
    brand_df["Qty_Impact_%"] - brand_df["Cost_Impact_%"]
).abs()

brand_df["level"] = "BRAND"


# 4. SKU IMPACT PRE-AGG (ONCE)


sku_impact = (
    sku_df
    .groupby(
        ["corp_brand_id", "corporate_brand", "date", "snapshot_date", "sku_id"],
        as_index=False
    )["abs_sku_impact"]
    .sum()
)

sku_impact = sku_impact.sort_values(
    ["corp_brand_id", "corporate_brand", "date", "snapshot_date", "abs_sku_impact"],
    ascending=[True, True, True, True, False]
)


# 5. TOP-K SKU IMPACTS (ONE PASS)

topk = (
    sku_impact
    .groupby(
        ["corp_brand_id", "corporate_brand", "date", "snapshot_date"]
    )
    .apply(
        lambda x: pd.Series({
            "Top1_SKU_Impact": x["abs_sku_impact"].head(1).sum(),
            "Top5_SKU_Impact": x["abs_sku_impact"].head(5).sum(),
            "Top10_SKU_Impact": x["abs_sku_impact"].head(10).sum(),
        })
    )
    .reset_index()
)

brand_df = brand_df.merge(
    topk,
    on=["corp_brand_id", "corporate_brand", "date", "snapshot_date"],
    how="left"
)

brand_df["Top_1_SKU_%"] = safe_div(
    brand_df["Top1_SKU_Impact"],
    brand_df["Abs_Impact"]
)

brand_df["Top_5_SKU_%"] = safe_div(
    brand_df["Top5_SKU_Impact"],
    brand_df["Abs_Impact"]
)

brand_df["Top_10_SKU_%"] = safe_div(
    brand_df["Top10_SKU_Impact"],
    brand_df["Abs_Impact"]
)


# 6. TOP PLANT CONCENTRATION

plant_impact = (
    sku_df
    .groupby(
        ["corp_brand_id", "corporate_brand", "date", "snapshot_date", "plant"],
        as_index=False
    )["abs_sku_impact"]
    .sum()
)

top_plant = (
    plant_impact
    .sort_values(
        ["corp_brand_id", "corporate_brand", "date", "snapshot_date", "abs_sku_impact"],
        ascending=[True, True, True, True, False]
    )
    .groupby(
        ["corp_brand_id", "corporate_brand", "date", "snapshot_date"]
    )
    .head(1)
    .rename(columns={"abs_sku_impact": "Top_Plant_Impact"})
)

brand_df = brand_df.merge(
    top_plant[
        ["corp_brand_id", "corporate_brand", "date", "snapshot_date", "Top_Plant_Impact"]
    ],
    on=["corp_brand_id", "corporate_brand", "date", "snapshot_date"],
    how="left"
)

brand_df["Plant_Concentration_%"] = safe_div(
    brand_df["Top_Plant_Impact"],
    brand_df["Abs_Impact"]
)

# 7. COMPOSITE SCORES

brand_df["Actionability_Index"] = (
    brand_df["Impact_from_Signal_%"] *
    brand_df["Top_5_SKU_%"]
)

brand_df["Explainability_Score"] = (
    0.4 * brand_df["Top_5_SKU_%"] +
    0.4 * brand_df["Signal_SKU_%"] +
    0.2 * (brand_df["Avg_Persistence"] / 6)
)


/tmp/ipykernel_393262/2950878673.py:118: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


In [49]:
brand_df.columns

Index(['corp_brand_id', 'corporate_brand', 'date', 'snapshot_date', 'ΔFIP',
       'Prior_FIP', 'Abs_Impact', 'Total_SKUs', 'Signal_SKUs', 'Signal_Impact',
       'Qty_Impact', 'Cost_Impact', 'Weighted_Dom_Num', 'Weighted_Dom_Den',
       'Driver_Conflict_Count', 'Avg_Persistence', 'Structural_CP_Count',
       'FIP_pct_change', 'Net_vs_Abs_Ratio', 'Signal_SKU_%',
       'Impact_from_Signal_%', 'Qty_Impact_%', 'Cost_Impact_%',
       'Weighted_Dominance', 'Driver_Conflict_%', 'Structural_Shift_Index',
       'Ownership_Clarity_Index', 'level', 'Top1_SKU_Impact',
       'Top5_SKU_Impact', 'Top10_SKU_Impact', 'Top_1_SKU_%', 'Top_5_SKU_%',
       'Top_10_SKU_%', 'Top_Plant_Impact', 'Plant_Concentration_%',
       'Actionability_Index', 'Explainability_Score'],
      dtype='object')

### Brand Level Calculations along with Dominant Level

In [78]:
# LEVEL_ORDER = [
#     "material_type",
#     "dosage_form_parent",
#     "material_group",
#     "material",
#     "nodetype",
#     "Plant Type",
#     "plant",
#     "material × plant"
# ]

# LEVEL_MAP = {
#     "material_type": {
#         "cols": ["material_type"],
#         "level_name": "BRAND_MATERIAL_TYPE"
#     },
#     "dosage_form_parent": {
#         "cols": ["dosage_form_parent"],
#         "level_name": "BRAND_DOSAGE_FORM"
#     },
#     "material_group": {
#         "cols": ["material_group"],
#         "level_name": "BRAND_MATERIAL_GROUP"
#     },
#     "material": {
#         "cols": ["material"],
#         "level_name": "BRAND_MATERIAL"
#     },
#     "nodetype": {
#         "cols": ["nodetype"],
#         "level_name": "BRAND_NODETYPE"
#     },
#     "Plant Type": {
#         "cols": ["Plant Type"],
#         "level_name": "BRAND_PLANT_TYPE"
#     },
#     "plant": {
#         "cols": ["plant"],
#         "level_name": "BRAND_PLANT"
#     },
#     "material × plant": {
#         "cols": ["material", "plant"],
#         "level_name": "BRAND_MATERIAL_PLANT"
#     }
# }


In [79]:
# def compute_entity_metrics(df):
#     out = {}

#     # Core aggregates
#     out["ΔFIP"] = df["total_fip_change"].sum()
#     out["Prior_FIP"] = df["total_cost_prev"].sum()
#     out["Abs_Impact"] = df["abs_sku_impact"].sum()

#     out["Total_SKUs"] = df["sku_id"].nunique()
#     out["Signal_SKUs"] = df["is_signal_governed"].sum()
#     out["Signal_Impact"] = df["signal_impact"].sum()

#     out["Qty_Impact"] = df["quantity_impact"].sum()
#     out["Cost_Impact"] = df["cost_impact"].sum()

#     # Weighted dominance
#     out["Weighted_Dominance"] = (
#         df["weighted_dom_component"].sum() / out["Abs_Impact"]
#         if out["Abs_Impact"] > 0 else np.nan
#     )

#     # Ratios
#     out["FIP_pct_change"] = (
#         out["ΔFIP"] / out["Prior_FIP"]
#         if out["Prior_FIP"] != 0 else np.nan
#     )

#     out["Net_vs_Abs_Ratio"] = (
#         abs(out["ΔFIP"]) / out["Abs_Impact"]
#         if out["Abs_Impact"] > 0 else np.nan
#     )

#     out["Signal_SKU_%"] = (
#         out["Signal_SKUs"] / out["Total_SKUs"]
#         if out["Total_SKUs"] > 0 else np.nan
#     )

#     out["Impact_from_Signal_%"] = (
#         out["Signal_Impact"] / out["Abs_Impact"]
#         if out["Abs_Impact"] > 0 else np.nan
#     )

#     out["Qty_Impact_%"] = (
#         out["Qty_Impact"] / out["ΔFIP"]
#         if out["ΔFIP"] != 0 else np.nan
#     )

#     out["Cost_Impact_%"] = (
#         out["Cost_Impact"] / out["ΔFIP"]
#         if out["ΔFIP"] != 0 else np.nan
#     )

#     out["Driver_Conflict_%"] = (
#         df["driver_conflict"].sum() / out["Signal_SKUs"]
#         if out["Signal_SKUs"] > 0 else np.nan
#     )

#     out["Avg_Persistence"] = df["quantity_persistence_score"].mean()

#     out["Structural_Shift_Index"] = (
#         (df["signal_structural_cp"].sum() / out["Signal_SKUs"]) * 100
#         if out["Signal_SKUs"] > 0 else np.nan
#     )

#     out["Ownership_Clarity_Index"] = abs(
#         out["Qty_Impact_%"] - out["Cost_Impact_%"]
#     )

#     return out


In [80]:
# # Concentration + HHI at a level
# def compute_level_concentration(sku_brand_df, group_cols, brand_delta_fip):

#     if brand_delta_fip == 0 or sku_brand_df.empty:
#         return None

#     grp = (
#         sku_brand_df
#         .groupby(group_cols, as_index=False)
#         .agg(
#             Group_ΔFIP=("total_fip_change", "sum"),
#             Abs_Impact=("abs_sku_impact", "sum"),
#             SKUs=("sku_id", "nunique")
#         )
#     )

#     grp["share"] = grp["Group_ΔFIP"].abs() / abs(brand_delta_fip)
#     grp = grp.sort_values("share", ascending=False)

#     max_share = grp["share"].iloc[0]
#     hhi = (grp["share"] ** 2).sum()

#     return {
#         "max_share": max_share,
#         "hhi": hhi,
#         "top_entity": grp[group_cols].iloc[0].to_dict(),
#         "top_skus": grp["SKUs"].iloc[0]
#     }

# # Dynamic level selection
# def select_winning_level(sku_brand_df, brand_delta_fip):

#     C_HIGH = 0.70
#     C_MODERATE = 0.50
#     H_CONCENTRATED = 0.60

#     best_candidate = None

#     for level_key in LEVEL_ORDER:
#         cfg = LEVEL_MAP[level_key]

#         res = compute_level_concentration(
#             sku_brand_df,
#             cfg["cols"],
#             brand_delta_fip
#         )

#         if res is None:
#             continue

#         if res["max_share"] > 0.95 and res["top_skus"] > 1:
#             continue

#         if res["max_share"] >= C_HIGH and res["hhi"] >= H_CONCENTRATED:
#             return {
#                 "level": level_key,
#                 "entity": res["top_entity"],
#                 "story_type": "SINGLE",
#                 "share": res["max_share"]
#             }

#         if res["max_share"] >= C_MODERATE:
#             if best_candidate is None or res["hhi"] > best_candidate["hhi"]:
#                 best_candidate = {
#                     "level": level_key,
#                     "entity": res["top_entity"],
#                     "hhi": res["hhi"],
#                     "share": res["max_share"]
#                 }

#     if best_candidate:
#         return {
#             "level": best_candidate["level"],
#             "entity": best_candidate["entity"],
#             "story_type": "MODERATE",
#             "share": best_candidate["share"]
#         }

#     return None

# # dominant entity rows
# dominant_rows = []

# for _, brand_row in brand_df.iterrows():

#     sku_brand_df = sku_df.loc[
#         (sku_df["corp_brand_id"] == brand_row["corp_brand_id"]) &
#         (sku_df["date"] == brand_row["date"]) &
#         (sku_df["snapshot_date"] == brand_row["snapshot_date"])
#     ]

#     if sku_brand_df.empty:
#         continue

#     selection = select_winning_level(
#         sku_brand_df,
#         brand_row["ΔFIP"]
#     )

#     if selection is None:
#         continue

#     cfg = LEVEL_MAP[selection["level"]]

#     entity_mask = (
#         (sku_df["corp_brand_id"] == brand_row["corp_brand_id"]) &
#         (sku_df["date"] == brand_row["date"]) &
#         (sku_df["snapshot_date"] == brand_row["snapshot_date"])
#     )

#     for c, v in selection["entity"].items():
#         entity_mask &= (sku_df[c] == v)

#     sub_df = sku_df.loc[entity_mask]
#     if sub_df.empty:
#         continue

#     metrics = compute_entity_metrics(sub_df)

#     metrics.update({
#         "corp_brand_id": brand_row["corp_brand_id"],
#         "corporate_brand": (
#             f"{brand_row['corporate_brand']}_"
#             + "_".join(selection["entity"].values())
#         ),
#         "date": brand_row["date"],
#         "snapshot_date": brand_row["snapshot_date"],
#         "level": cfg["level_name"],
#         "Entity_Abs_Impact_Share": (
#             metrics["Abs_Impact"] / brand_row["Abs_Impact"]
#             if brand_row["Abs_Impact"] != 0 else np.nan
#         ),
#         "Story_Type": selection["story_type"]
#     })

#     dominant_rows.append(metrics)

# # final df

# dominant_entity_df = pd.DataFrame(dominant_rows)

# final_agg_df = pd.concat(
#     [brand_df, dominant_entity_df],
#     ignore_index=True
# )

In [93]:
HIERARCHY = [
    ("material_type", ["material_type"]),
    ("dosage_form_parent", ["dosage_form_parent"]),
    ("material_group", ["material_group"]),
    ("material", ["material"]),
    ("Plant Type", ["Plant Type"]),
    ("plant", ["plant"])
]

In [94]:
def safe_div(n, d):
    return np.where(d != 0, n / d, np.nan)


brand_df = (
    sku_df
    .groupby(
        ["corp_brand_id", "corporate_brand", "date", "snapshot_date"],
        as_index=False
    )
    .agg(
        ΔFIP=("total_fip_change", "sum"),
        Prior_FIP=("total_cost_prev", "sum"),
        Abs_Impact=("abs_sku_impact", "sum"),

        Total_SKUs=("sku_id", "nunique"),
        Signal_SKUs=("is_signal_governed", "sum"),
        Signal_Impact=("signal_impact", "sum"),

        Qty_Impact=("quantity_impact", "sum"),
        Cost_Impact=("cost_impact", "sum"),

        Weighted_Dom_Num=("weighted_dom_component", "sum"),
        Weighted_Dom_Den=("abs_sku_impact", "sum"),

        Driver_Conflict_Count=("driver_conflict", "sum"),
        Avg_Persistence=("quantity_persistence_score", "mean"),
        Structural_CP_Count=("signal_structural_cp", "sum"),
    )
)

brand_df["FIP_pct_change"] = safe_div(brand_df["ΔFIP"], brand_df["Prior_FIP"])
brand_df["Net_vs_Abs_Ratio"] = safe_div(abs(brand_df["ΔFIP"]), brand_df["Abs_Impact"])

brand_df["Signal_SKU_%"] = safe_div(
    brand_df["Signal_SKUs"], brand_df["Total_SKUs"]
)

brand_df["Impact_from_Signal_%"] = safe_div(
    brand_df["Signal_Impact"], brand_df["Abs_Impact"]
)

brand_df["Qty_Impact_%"] = safe_div(
    brand_df["Qty_Impact"], brand_df["ΔFIP"]
)

brand_df["Cost_Impact_%"] = safe_div(
    brand_df["Cost_Impact"], brand_df["ΔFIP"]
)

brand_df["Weighted_Dominance"] = safe_div(
    brand_df["Weighted_Dom_Num"], brand_df["Weighted_Dom_Den"]
)

brand_df["Driver_Conflict_%"] = safe_div(
    brand_df["Driver_Conflict_Count"], brand_df["Signal_SKUs"]
)

brand_df["Structural_Shift_Index"] = safe_div(
    brand_df["Structural_CP_Count"], brand_df["Signal_SKUs"]
) * 100

brand_df["Ownership_Clarity_Index"] = (
    brand_df["Qty_Impact_%"] - brand_df["Cost_Impact_%"]
).abs()

brand_df["level"] = "BRAND"



In [98]:
def compute_level_concentration(sku_brand_df, group_cols, brand_delta_fip):

    if sku_brand_df.empty or brand_delta_fip == 0:
        return None

    grp = (
        sku_brand_df
        .groupby(group_cols, as_index=False)
        .agg(Group_ΔFIP=("total_fip_change", "sum"))
    )

    grp["share"] = grp["Group_ΔFIP"].abs() / abs(brand_delta_fip)
    grp = grp.sort_values("share", ascending=False)

    hhi = (grp["share"] ** 2).sum()
    top2_coverage = grp["share"].head(2).sum()

    return {
        "max_share": grp["share"].iloc[0],
        "hhi": hhi,
        "top_entity": grp[group_cols].iloc[0].to_dict(),
        "top2_entities": grp[group_cols].head(2).to_dict("records"),
        "top2_coverage": top2_coverage
    }


def select_winning_level(sku_brand_df, brand_delta_fip):

    C_HIGH = 0.70
    C_MODERATE = 0.50
    H_CONCENTRATED = 0.60

    best = None

    for level_name, cols in HIERARCHY:

        res = compute_level_concentration(
            sku_brand_df, cols, brand_delta_fip
        )
        if res is None:
            continue

        max_share = res["max_share"]
        hhi = res["hhi"]

        # Strong dominance
        if max_share >= C_HIGH and hhi >= H_CONCENTRATED:

            if best is None:
                best = {
                    "level": level_name,
                    "cols": cols,
                    "entity": res["top_entity"],
                    "share": max_share,
                    "hhi": hhi,
                    "Concentration_Regime": "Single dominant"
                }
                continue

            if hhi > best["hhi"]:
                best.update({
                    "level": level_name,
                    "cols": cols,
                    "entity": res["top_entity"],
                    "share": max_share,
                    "hhi": hhi
                })
                continue

            return best  # clarity dropped → revert

        # Moderate dominance
        if max_share >= C_MODERATE:
            if best is None or hhi > best["hhi"]:
                best = {
                    "level": level_name,
                    "cols": cols,
                    "entity": res["top_entity"],
                    "share": max_share,
                    "hhi": hhi,
                    "Concentration_Regime": "Moderate dominant"
                }

    if best:
        return best

    # Distributed fallback
    fallback = compute_level_concentration(
    sku_brand_df, ["material_type"], brand_delta_fip)

    if fallback is None:
        return {
        "Concentration_Regime": "Structurally concentrated",
        "reason": "Single group dominates across all hierarchy levels"
        }

    return {
    "level": "material_type",
    "cols": ["material_type"],
    "entities": fallback["top2_entities"],
    "share": fallback["top2_coverage"],
    "Concentration_Regime": "Distributed"
    }
    

#compute metrics
def compute_entity_metrics(df):

    out = {}

    out["ΔFIP"] = df["total_fip_change"].sum()
    out["Prior_FIP"] = df["total_cost_prev"].sum()
    out["Abs_Impact"] = df["abs_sku_impact"].sum()

    out["Total_SKUs"] = df["sku_id"].nunique()
    out["Signal_SKUs"] = df["is_signal_governed"].sum()
    out["Signal_Impact"] = df["signal_impact"].sum()

    out["Qty_Impact"] = df["quantity_impact"].sum()
    out["Cost_Impact"] = df["cost_impact"].sum()

    out["Weighted_Dominance"] = safe_div(
        df["weighted_dom_component"].sum(),
        df["abs_sku_impact"].sum()
    )

    out["Signal_SKU_%"] = safe_div(out["Signal_SKUs"], out["Total_SKUs"])
    out["Impact_from_Signal_%"] = safe_div(out["Signal_Impact"], out["Abs_Impact"])
    out["Qty_Impact_%"] = safe_div(out["Qty_Impact"], out["ΔFIP"])
    out["Cost_Impact_%"] = safe_div(out["Cost_Impact"], out["ΔFIP"])

    out["Avg_Persistence"] = df["quantity_persistence_score"].mean()

    out["Structural_Shift_Index"] = safe_div(
        df["signal_structural_cp"].sum(),
        out["Signal_SKUs"]
    ) * 100

    out["Ownership_Clarity_Index"] = abs(
        out["Qty_Impact_%"] - out["Cost_Impact_%"]
    )

    return out

# final df
dominant_rows = []

for _, brand_row in brand_df.iterrows():

    sku_brand_df = sku_df.loc[
        (sku_df["corp_brand_id"] == brand_row["corp_brand_id"]) &
        (sku_df["date"] == brand_row["date"]) &
        (sku_df["snapshot_date"] == brand_row["snapshot_date"])
    ]

    if sku_brand_df.empty:
        continue

    selection = select_winning_level(
        sku_brand_df,
        brand_row["ΔFIP"]
    )

    if selection is None or "entity" not in selection:
        continue

    entity_mask = (
        (sku_df["corp_brand_id"] == brand_row["corp_brand_id"]) &
        (sku_df["date"] == brand_row["date"]) &
        (sku_df["snapshot_date"] == brand_row["snapshot_date"])
    )

    for col, val in selection["entity"].items():
        entity_mask &= (sku_df[col] == val)

    sub_df = sku_df.loc[entity_mask]
    if sub_df.empty:
        continue

    metrics = compute_entity_metrics(sub_df)

    metrics.update({
        "corp_brand_id": brand_row["corp_brand_id"],
        "corporate_brand": (
            f"{brand_row['corporate_brand']}_"
            + "_".join(selection["entity"].values())
        ),
        "date": brand_row["date"],
        "snapshot_date": brand_row["snapshot_date"],
        "level": f"BRAND_{selection['level'].upper()}",
        "Entity_Impact_onBrandlevel_%": (
            abs(metrics["ΔFIP"]) / abs(brand_row["ΔFIP"])
            if brand_row["ΔFIP"] != 0 else np.nan
        ),
        "Concentration_Regime": selection["Concentration_Regime"]
    })

    dominant_rows.append(metrics)


dominant_entity_df = pd.DataFrame(dominant_rows)

final_agg_df = pd.concat(
    [brand_df, dominant_entity_df],
    ignore_index=True
)


In [99]:
final_agg_df[final_agg_df['snapshot_date'] == '2026-01-09']

,corp_brand_id,corporate_brand,date,snapshot_date,ΔFIP,Prior_FIP,Abs_Impact,Total_SKUs,Signal_SKUs,Signal_Impact,...,Impact_from_Signal_%,Qty_Impact_%,Cost_Impact_%,Weighted_Dominance,Driver_Conflict_%,Structural_Shift_Index,Ownership_Clarity_Index,level,Entity_Impact_onBrandlevel_%,Concentration_Regime
16,00201790,All Other Pharmaceut,202612,2026-01-09,-1.189476e+07,1.474060e+08,1.372130e+07,2150,1062,1.363193e+07,...,0.993487,1.0,-0.0,1.0,0.212806,0.094162,1.0,BRAND,NaN,NaN
37,03302043,ABRAXANE,202612,2026-01-09,-7.654394e+05,1.666681e+08,4.953346e+06,130,103,1.889120e+06,...,0.381383,1.0,-0.0,1.0,0.165049,0.000000,1.0,BRAND,NaN,NaN
58,03302101,REVLIMID,202612,2026-01-09,-1.491973e+09,1.523524e+09,1.493309e+09,808,694,3.282400e+06,...,0.002198,1.0,-0.0,1.0,0.170029,0.288184,1.0,BRAND,NaN,NaN
76,00201790,All Other Pharmaceut_nan,202612,2026-01-09,-1.189476e+07,1.447419e+08,1.372130e+07,2131,1052,1.363193e+07,...,0.9934867933765065,1.0,-0.0,1.0,NaN,0.095057,1.0,BRAND_DOSAGE_FORM_PARENT,1.000000,Single dominant
94,03302043,ABRAXANE_HALB,202612,2026-01-09,-1.448264e+06,6.586077e+07,3.777557e+06,15,13,1.165168e+06,...,0.3084447518732015,1.0,-0.0,1.0,NaN,0.000000,1.0,BRAND_MATERIAL_TYPE,1.892069,Single dominant
112,03302101,REVLIMID_HALB,202612,2026-01-09,-1.485717e+09,1.493397e+09,1.486367e+09,109,89,1.149329e+06,...,0.0007732471689655149,1.0,-0.0,1.0,NaN,0.000000,1.0,BRAND_MATERIAL_TYPE,0.995807,Single dominant


In [85]:
final_agg_df.columns

Index(['corp_brand_id', 'corporate_brand', 'date', 'snapshot_date', 'ΔFIP',
       'Prior_FIP', 'Abs_Impact', 'Total_SKUs', 'Signal_SKUs', 'Signal_Impact',
       'Qty_Impact', 'Cost_Impact', 'Weighted_Dom_Num', 'Weighted_Dom_Den',
       'Driver_Conflict_Count', 'Avg_Persistence', 'Structural_CP_Count',
       'FIP_pct_change', 'Net_vs_Abs_Ratio', 'Signal_SKU_%',
       'Impact_from_Signal_%', 'Qty_Impact_%', 'Cost_Impact_%',
       'Weighted_Dominance', 'Driver_Conflict_%', 'Structural_Shift_Index',
       'Ownership_Clarity_Index', 'level', 'Top1_SKU_Impact',
       'Top5_SKU_Impact', 'Top10_SKU_Impact', 'Top_1_SKU_%', 'Top_5_SKU_%',
       'Top_10_SKU_%', 'Top_Plant_Impact', 'Plant_Concentration_%',
       'Actionability_Index', 'Explainability_Score',
       'Entity_Abs_Impact_Share', 'Story_Type'],
      dtype='object')

In [88]:
# Select & rename final columns

final_df = (
    final_agg_df
    .rename(columns={
        "corporate_brand": "entity",
        "FIP_pct_change": "FIP_change_pct",
        "Entity_Abs_Impact_Share": "Entity_Abs_Impact_Share_pct",
        "Net_vs_Abs_Ratio": "Net_vs_Abs_pct",
        "Signal_SKU_%": "Signal_SKU_pct",
        "Impact_from_Signal_%": "Impact_from_Signal_pct",
        "Top_1_SKU_%": "Top_1_SKU_pct",
        "Top_5_SKU_%": "Top_5_SKU_pct",
        "Top_10_SKU_%": "Top_10_SKU_pct",
        "Plant_Concentration_%": "Plant_Concentration_pct",
        "Qty_Impact_%": "Qty_Impact_pct",
        "Cost_Impact_%": "Cost_Impact_pct",
        "Driver_Conflict_%": "Driver_Conflict_pct",
        "Actionability_Index": "Actionability_Index_pct",
        "Ownership_Clarity_Index": "Ownership_Clarity_Index_pct",
        "Explainability_Score": "Explainability_Score_pct",
        "Weighted_Dominance": "Driver_Weighted_Dominance",
        "Structural_Shift_Index": "Structural_Shift_Index_pct",
        "Story_Type" : "dominance_pattern"
    })
    [
        [
            "corp_brand_id",
            "entity",
            "level",
            "date",
            "snapshot_date",
            "FIP_change_pct",
            "Abs_Impact",
            "Entity_Abs_Impact_Share_pct",
            "Net_vs_Abs_pct",
            "Signal_SKU_pct",
            "Impact_from_Signal_pct",
            "Top_1_SKU_pct",
            "Top_5_SKU_pct",
            "Top_10_SKU_pct",
            "Plant_Concentration_pct",
            "Qty_Impact_pct",
            "Cost_Impact_pct",
            "Driver_Weighted_Dominance",
            "Driver_Conflict_pct",
            "Avg_Persistence",
            "Structural_Shift_Index_pct",
            "Actionability_Index_pct",
            "Ownership_Clarity_Index_pct",
            "Explainability_Score_pct",
            "dominance_pattern"
            
        ]
    ]
)


# Percentage columns (×100)
percent_cols = [
    "FIP_change_pct",
    "Entity_Abs_Impact_Share_pct",
    "Net_vs_Abs_pct",
    "Signal_SKU_pct",
    "Impact_from_Signal_pct",
    "Top_1_SKU_pct",
    "Top_5_SKU_pct",
    "Top_10_SKU_pct",
    "Plant_Concentration_pct",
    "Qty_Impact_pct",
    "Cost_Impact_pct",
    "Driver_Conflict_pct",
    "Actionability_Index_pct",
    "Ownership_Clarity_Index_pct",
    "Explainability_Score_pct",
]

final_df[percent_cols] = (
    final_df[percent_cols]
    .astype(float)
    .mul(100)
    .round(2)
)


# Numeric rounding

# Round most numeric columns to 2 decimals
numeric_round_2_cols = [
    "Abs_Impact",
    "Avg_Persistence",
    "Structural_Shift_Index_pct",
]

final_df[numeric_round_2_cols] = (
    final_df[numeric_round_2_cols]
    .astype(float)
    .round(2)
)

# Round Driver_Weighted_Dominance to 0 decimals
final_df["Driver_Weighted_Dominance"] = (
    final_df["Driver_Weighted_Dominance"]
    .astype(float)
    .round(0)
    .astype("Int64")
)


def remove_negative_zero(x):
    if isinstance(x, (int, float, np.floating)) and np.isclose(x, 0):
        return 0.0
    return x

cols_to_clean = [
    "Qty_Impact_pct",
    "Cost_Impact_pct",
    "Net_vs_Abs_pct",
    "Driver_Conflict_pct",
]

final_df[cols_to_clean] = final_df[cols_to_clean].applymap(remove_negative_zero)


/tmp/ipykernel_393262/912776497.py:122: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  final_df[cols_to_clean] = final_df[cols_to_clean].applymap(remove_negative_zero)


In [89]:
final_df.columns

Index(['corp_brand_id', 'entity', 'level', 'date', 'snapshot_date',
       'FIP_change_pct', 'Abs_Impact', 'Entity_Abs_Impact_Share_pct',
       'Net_vs_Abs_pct', 'Signal_SKU_pct', 'Impact_from_Signal_pct',
       'Top_1_SKU_pct', 'Top_5_SKU_pct', 'Top_10_SKU_pct',
       'Plant_Concentration_pct', 'Qty_Impact_pct', 'Cost_Impact_pct',
       'Driver_Weighted_Dominance', 'Driver_Conflict_pct', 'Avg_Persistence',
       'Structural_Shift_Index_pct', 'Actionability_Index_pct',
       'Ownership_Clarity_Index_pct', 'Explainability_Score_pct',
       'dominance_pattern'],
      dtype='object')

In [92]:
final_df[(final_df['dominance_pattern'] != 'SINGLE') & (final_df['level'] != 'BRAND') ]

,corp_brand_id,entity,level,date,snapshot_date,FIP_change_pct,Abs_Impact,Entity_Abs_Impact_Share_pct,Net_vs_Abs_pct,Signal_SKU_pct,...,Qty_Impact_pct,Cost_Impact_pct,Driver_Weighted_Dominance,Driver_Conflict_pct,Avg_Persistence,Structural_Shift_Index_pct,Actionability_Index_pct,Ownership_Clarity_Index_pct,Explainability_Score_pct,dominance_pattern
77,00201790,All Other Pharmaceut_Internal,BRAND_PLANT_TYPE,202612,2026-01-16,-13.27,25926619.13,74.48,42.51,50.67,...,100.0,0.0,1,45.74,0.41,0.63,NaN,100.0,NaN,MODERATE
113,03302101,REVLIMID_FIN,BRAND_MATERIAL_TYPE,202612,2026-01-16,24.21,6158423.63,49.89,90.09,92.93,...,100.0,0.0,1,17.07,0.23,0.00,NaN,100.0,NaN,MODERATE


In [69]:
DEBUG_BRAND = "REVLIMID"
DEBUG_DATE = "202612"
DEBUG_SNAPSHOT = "2026-01-09"

sku_dbg = sku_df[
    (sku_df["corporate_brand"] == DEBUG_BRAND) &
    (sku_df["date"] == DEBUG_DATE) &
    (sku_df["snapshot_date"] == DEBUG_SNAPSHOT)
].copy()

brand_fip = sku_dbg["total_fip_change"].sum()

print("Brand ΔFIP:", brand_fip)
print("SKU count:", sku_dbg["sku_id"].nunique())


Brand ΔFIP: -1491973166.787145
SKU count: 808


In [70]:
def debug_level(sku_brand_df, group_cols, brand_fip):
    grp = (
        sku_brand_df
        .groupby(group_cols, as_index=False)
        .agg(
            Group_ΔFIP=("total_fip_change", "sum"),
            Abs_Impact=("abs_sku_impact", "sum"),
            SKUs=("sku_id", "nunique")
        )
    )

    grp["share"] = np.where(
        brand_fip != 0,
        grp["Group_ΔFIP"].abs() / abs(brand_fip),
        np.nan
    )

    grp = grp.sort_values("share", ascending=False)

    hhi = (grp["share"] ** 2).sum()
    max_share = grp["share"].iloc[0] if not grp.empty else 0

    return grp, {
        "max_share": max_share,
        "hhi": hhi,
        "top_group": grp[group_cols].iloc[0].to_dict() if not grp.empty else {}
    }


DEBUG_HIERARCHY = [
    ["material_type"],
    ["dosage_form_parent"],
    ["material_group"],
    ["material"],
    ["nodetype"],
    ["Plant Type"],
    ["plant"],
    ["material", "plant"]
]

for level in DEBUG_HIERARCHY:
    print("\n" + "="*80)
    print("LEVEL:", " × ".join(level))

    grp, stats = debug_level(sku_dbg, level, brand_fip)

    display(
        grp.head(10)  # top contributors
    )

    print("Max Share:", round(stats["max_share"], 3))
    print("HHI:", round(stats["hhi"], 3))
    print("Top Entity:", stats["top_group"])

    # Explain decision
    if stats["max_share"] > 0.95:
        print("❌ SKIPPED → Non-discriminating (>95%)")
    elif stats["max_share"] >= 0.70 and stats["hhi"] >= 0.60:
        print("✅ STRONG SINGLE-GROUP STORY")
    elif stats["max_share"] >= 0.50:
        print("⚠️ MODERATE CANDIDATE")
    else:
        print("❌ DISTRIBUTED / WEAK")



LEVEL: material_type


,material_type,Group_ΔFIP,Abs_Impact,SKUs,share
1,HALB,-1.485717e+09,1.486367e+09,109,0.995807
0,FIN,-6.245475e+06,6.865492e+06,581,0.004186
3,RAW,-2.307534e+04,2.307534e+04,8,0.000015
2,PACK,1.244402e+04,5.403471e+04,110,0.000008


Max Share: 0.996
HHI: 0.992
Top Entity: {'material_type': 'HALB'}
❌ SKIPPED → Non-discriminating (>95%)

LEVEL: dosage_form_parent


,dosage_form_parent,Group_ΔFIP,Abs_Impact,SKUs,share
1,nan,-1.485392e+09,1.485539e+09,125,0.995589
0,CAPSULE,-6.581070e+06,7.770584e+06,683,0.004411


Max Share: 0.996
HHI: 0.991
Top Entity: {'dosage_form_parent': 'nan'}
❌ SKIPPED → Non-discriminating (>95%)

LEVEL: material_group


...

Max Share: 0.995
HHI: 0.991
Top Entity: {'Plant Type': 'External'}
❌ SKIPPED → Non-discriminating (>95%)

LEVEL: plant


,plant,Group_ΔFIP,Abs_Impact,SKUs,share
38,2091,-1.485004e+09,1.485222e+09,100,0.995329
35,2061,-1.829386e+06,2.281429e+06,219,0.001226
39,2093,-1.129199e+06,1.310109e+06,68,0.000757
16,1763,-9.368249e+05,9.368249e+05,8,0.000628
8,1727,-8.515596e+05,8.515596e+05,11,0.000571
40,2121,-5.467795e+05,7.325017e+05,46,0.000366
2,1609,-3.968258e+05,3.968258e+05,4,0.000266
23,2027,-3.608869e+05,3.608869e+05,5,0.000242
21,2023,-3.064237e+05,3.064237e+05,27,0.000205
5,1724,-2.682599e+05,2.682599e+05,2,0.000180


Max Share: 0.995
HHI: 0.991
Top Entity: {'plant': '2091'}
❌ SKIPPED → Non-discriminating (>95%)

LEVEL: material × plant


,material,plant,Group_ΔFIP,Abs_Impact,SKUs,share
645,1456877,2091,-1.482382e+09,1.482382e+09,1,0.993571
643,1455892,2091,-2.654674e+06,2.654674e+06,1,0.001779
682,1461862,1727,-8.515596e+05,8.515596e+05,1,0.000571
180,1431533,1763,-5.051072e+05,5.051072e+05,1,0.000339
619,1443737,2093,-4.321820e+05,4.321820e+05,1,0.000290
195,1431551,1609,-3.910391e+05,3.910391e+05,1,0.000262
642,1455892,2061,-3.852802e+05,3.852802e+05,1,0.000258
220,1431567,1763,-3.409686e+05,3.409686e+05,1,0.000229
616,1443737,2027,-3.267963e+05,3.267963e+05,1,0.000219
614,1443737,2023,-3.064237e+05,3.064237e+05,1,0.000205


Max Share: 0.994
HHI: 0.987
Top Entity: {'material': '1456877', 'plant': '2091'}
❌ SKIPPED → Non-discriminating (>95%)


In [350]:
# final_agg_df.to_csv('3brands_agg_file.csv')

In [421]:
# brands_to_keep = [
#     "All Other Pharmaceut"
#      ,
#     "ABRAXANE"
#     ,
#     "REVLIMID"
# ]
# dates_to_keep = ["202612", "202712", "202812"]

# fil = final_agg_df[
#     final_agg_df["corporate_brand"].isin(brands_to_keep) &
#     final_agg_df["date"].isin(dates_to_keep)
# ]

In [425]:
# final_agg_df.to_csv('Allbrands_agg_file.csv')